# ⚙️ 02 — Pipeline de Preprocesamiento Hermético (Cero Data Leakage)
### Proyecto Final: Predicción de Severidad de Siniestros Viales en Medellín
**Especialización en Analítica y Ciencia de Datos / Machine Learning**

---

## 1. Fundamento Metodológico: Prevención de Fuga de Datos (Data Leakage)

Uno de los errores metodológicos más frecuentes en proyectos de Machine Learning es aplicar transformaciones globales (como imputar nulos con la media/mediana general, o ajustar un `StandardScaler` o `OneHotEncoder` sobre todo el dataset) **antes** de separar los datos en conjuntos de entrenamiento y prueba.

### ¿Por qué es un error grave?
Al calcular estadísticas sobre todo el dataset, el conjunto de prueba contamina el entrenamiento con información del futuro (*Data Leakage*), produciendo métricas de evaluación artificialmente optimistas que fallan estrepitosamente en producción.

### La Solución de Ingeniería:
Encapsular todo el flujo de transformación dentro de un **`ColumnTransformer`** y **`Pipeline`** de Scikit-Learn. El pipeline se ajusta (`fit`) **estrictamente con `X_train`** y solo se aplica (`transform`) sobre `X_test`.

In [ ]:
# Importaciones modulares y configuración del entorno
import sys
from pathlib import Path

# Garantizar acceso al paquete modular src desde la carpeta notebooks
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import RANDOM_STATE, TEST_SIZE, BINARY_TARGET_COLUMN
from src.data.load_data import load_raw_data
from src.features.build_features import prepare_full_dataset
from src.features.transformers import RareCategoryGrouper
from src.models.pipeline import (
    build_preprocessor,
    get_feature_names,
    DEFAULT_NUMERICAL_FEATURES,
    DEFAULT_CATEGORICAL_FEATURES,
)

print(f"[OK] Módulos de pipeline importados exitosamente. Semilla: {RANDOM_STATE}")

## 2. Carga y Preparación de Variables Predictoras ($X$) y Target ($y$)

Cargamos la muestra y aplicamos la ingeniería de características base generada en `src.features.build_features`.

In [ ]:
df_raw = load_raw_data(use_sample=True)
df_clean = prepare_full_dataset(df_raw)

# Separación de variables predictoras (X) y variable objetivo (y)
feature_columns = DEFAULT_NUMERICAL_FEATURES + DEFAULT_CATEGORICAL_FEATURES
X = df_clean[feature_columns].copy()
y = df_clean[BINARY_TARGET_COLUMN].copy()

print(f"Conjunto X: {X.shape[0]:,} filas x {X.shape[1]} columnas predictoras")
print(f"Conjunto y: {len(y):,} etiquetas de severidad")
X.head(3)

## 3. Partición Estratificada en Entrenamiento y Prueba (Train / Test Split)

Utilizamos `stratify=y` para asegurar que la proporción exacta de incidentes graves (56.4%) y leves (43.6%) se mantenga idéntica en ambos subconjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Entrenamiento (X_train): {X_train.shape[0]:,} filas ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Prueba        (X_test):  {X_test.shape[0]:,} filas ({X_test.shape[0]/len(X)*100:.1f}%)")
print("\nDistribución porcentual del Target en Train vs Test:")
balance_check = pd.DataFrame({
    "Train (%)": (y_train.value_counts(normalize=True) * 100).round(2),
    "Test (%)": (y_test.value_counts(normalize=True) * 100).round(2),
})
balance_check

## 4. Estructura y Ensamblado del `ColumnTransformer`

Construimos el preprocesador modular mediante `build_preprocessor()`:
- **Rama Numérica:**
  1. `SimpleImputer(strategy='median')`: Imputa nulos numéricos usando la mediana de `X_train`.
  2. `StandardScaler()`: Estandariza a media 0 y varianza unitaria.
- **Rama Categórica:**
  1. `SimpleImputer(strategy='constant', fill_value='DESCONOCIDO')`: Maneja categorías vacías.
  2. `RareCategoryGrouper(min_freq=0.01)`: Agrupa categorías con menos del 1% de frecuencia en `OTRO`.
  3. `OneHotEncoder(handle_unknown='ignore')`: Binariza variables categóricas ignorando categorías nuevas no vistas en train.

In [ ]:
preprocessor = build_preprocessor()
preprocessor

## 5. Ajuste (`fit`) en Train y Transformación de Ambos Conjuntos

Demostración de ejecución hermética: `fit_transform` exclusivamente sobre `X_train`.

In [ ]:
# Ajuste exclusivo sobre X_train
X_train_proc = preprocessor.fit_transform(X_train)

# Transformación sobre X_test utilizando los parámetros aprendidos en train
X_test_proc = preprocessor.transform(X_test)

print(f"Matriz X_train_proc: {X_train_proc.shape[0]:,} filas x {X_train_proc.shape[1]} características generadas")
print(f"Matriz X_test_proc:  {X_test_proc.shape[0]:,} filas x {X_test_proc.shape[1]} características generadas")

### 5.1 Nombres de Columnas Resultantes

Obtenemos los nombres de las 50 características finales para futura interpretabilidad y análisis de coeficientes.

In [ ]:
final_features = get_feature_names(preprocessor)
print(f"Total características: {len(final_features)}")
print("Primeras 15 características generadas:")
for i, f in enumerate(final_features[:15], 1):
    print(f"{i:2d}. {f}")

## 6. Verificación Empírica de Cero Data Leakage

Comprobamos que las medias y desviaciones de estandarización provienen estrictamente de `X_train`:

In [ ]:
num_scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]
num_imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]

leakage_audit = pd.DataFrame({
    "Variable": DEFAULT_NUMERICAL_FEATURES,
    "Mediana aprendida en Train": num_imputer.statistics_,
    "Media de escalado (Train)": num_scaler.mean_.round(3),
    "Escala std (Train)": num_scaler.scale_.round(3),
})
leakage_audit

## 7. Conclusiones y Transición a la Fase 4 (Modelado)

1. **Garantía Metodológica:** El preprocesador queda 100% aislado, reutilizable y listo para ser integrado en `Pipeline(steps=[('preprocessor', preprocessor), ('model', clasificador)])`.
2. **Dimensionalidad Controlada:** La combinación de `RareCategoryGrouper` y `OneHotEncoder` genera exactamente 50 columnas densas y representativas sin explosión dimensional.
3. **Próximo Paso:** En la **Fase 4**, entrenaremos y compararemos los modelos: Baseline (Dummy/LogisticRegression), Random Forest y XGBoost, optimizando hiperparámetros y evaluando con PR-AUC y F1-Score.